In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import joblib

import nltk
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

# 2. Load the Spanish stop words
stop_words_es = set(stopwords.words('spanish'))

[nltk_data] Downloading package punkt to /Users/samir/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /Users/samir/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Preparación de datos

In [5]:
data = pd.read_excel('datos_miniproyecto2.xlsx')

In [6]:
data.head()

,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5


In [7]:
data.duplicated().sum()

np.int64(0)

In [8]:
data.isna().sum()

textos    0
ODS       0
dtype: int64

In [9]:
X = data['textos']
y = data['ODS']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
# Pre-load stopwords into a plain Python set BEFORE any parallel/joblib usage.
# Calling stopwords.words() inside the function causes NLTK LazyCorpusLoader
# pickling errors when GridSearchCV uses n_jobs=-1 (joblib multiprocessing).
_stop_words_es = set(stopwords.words("spanish"))
_tokenizer = RegexpTokenizer(r"\w+")
_stemmer = PorterStemmer()

def text_preprocess(text):
    tokens = _tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in _stop_words_es]
    tokens = [_stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

In [11]:
vectorizer = TfidfVectorizer(preprocessor=text_preprocess)

In [ ]:
output = vectorizer.fit_transform(X)
vectorizer.get_feature_names_out()

array(['00', '000', '0000002', ..., 'útero', 'útil', 'útile'],
      dtype=object)

In [12]:
tsvd = TruncatedSVD(n_components=10, random_state=42)

# Tópicos

In [ ]:
tsvd_output = tsvd.fit_transform(output)
tsvd_output

array([[ 0.09316818,  0.01770481, -0.00493943, ...,  0.02049497,
         0.00454611,  0.01394069],
       [ 0.21269731, -0.26595617,  0.1928858 , ...,  0.07717295,
         0.07350565, -0.02865811],
       [ 0.1525586 , -0.09539048,  0.03355576, ..., -0.08781662,
        -0.03574603,  0.01814264],
       ...,
       [ 0.11198195, -0.01988611,  0.00232823, ...,  0.01541555,
         0.05104164, -0.05608823],
       [ 0.16373775, -0.08872858,  0.05549034, ..., -0.03220087,
        -0.00729286, -0.01089059],
       [ 0.10709916,  0.00285441, -0.01299018, ...,  0.00912048,
         0.01615735,  0.00823297]])

In [ ]:
vocabulario = vectorizer.get_feature_names_out()
loadings = tsvd.components_
# --- [PASO 3] Identificar las palabras con mayor peso para 5 componentes ---
n_palabras_por_topico = 10  # Número de palabras principales a mostrar
componentes_a_mostrar = 5   # Al menos 5 componentes

print("=== EXTRACCIÓN DE TÓPICOS (PALABRAS CON MAYOR PESO) ===")
for i in range(componentes_a_mostrar):
    # Obtener los índices de los pesos ordenados de menor a mayor, y tomar los últimos (los más grandes)
    indices_palabras_top = np.argsort(loadings[i])[::-1][:n_palabras_por_topico]

    # Mapear esos índices a las palabras reales
    palabras_top = [vocabulario[idx] for idx in indices_palabras_top]
    pesos_top = [loadings[i][idx] for idx in indices_palabras_top]

    print(f"\n🔹 Componente / Tópico {i + 1}:")
    for palabra, peso in zip(palabras_top, pesos_top):
        print(f"  - {palabra}: {peso:.4f}")

=== EXTRACCIÓN DE TÓPICOS (PALABRAS CON MAYOR PESO) ===

🔹 Componente / Tópico 1:
  - mujer: 0.1578
  - la: 0.1546
  - país: 0.1409
  - política: 0.1338
  - agua: 0.1303
  - desarrollo: 0.1155
  - el: 0.1136
  - derecho: 0.1134
  - en: 0.1131
  - nivel: 0.1011

🔹 Componente / Tópico 2:
  - mujer: 0.4580
  - derecho: 0.3364
  - género: 0.2430
  - hombr: 0.1586
  - humano: 0.1259
  - igualdad: 0.1197
  - trabajo: 0.0929
  - internacion: 0.0839
  - violencia: 0.0808
  - artículo: 0.0654

🔹 Componente / Tópico 3:
  - derecho: 0.5420
  - agua: 0.2360
  - humano: 0.2163
  - internacion: 0.1850
  - artículo: 0.1275
  - ley: 0.0886
  - penal: 0.0764
  - internacional: 0.0740
  - est: 0.0727
  - tratado: 0.0637

🔹 Componente / Tópico 4:
  - agua: 0.5441
  - mujer: 0.4587
  - género: 0.2310
  - hombr: 0.1644
  - igualdad: 0.1000
  - subterránea: 0.0814
  - residual: 0.0601
  - hídrico: 0.0550
  - violencia: 0.0538
  - riego: 0.0465

🔹 Componente / Tópico 5:
  - pobreza: 0.3302
  - ingreso: 0.219

# Creación del pipeline y modelo de clasificación

In [ ]:
model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced')

In [14]:
steps = [
    ("vectorizer", vectorizer),
    ("dimred", tsvd),
    ("model", model),
]
pipeline = Pipeline(steps)

In [ ]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__max_iter': [500, 1000]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=2)

In [16]:
grid_search.fit(X_train, y_train)

/Users/samir/Desktop/proyectos/maestria/MiniProjecto2-NoSupervisado/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/samir/Desktop/proyectos/maestria/MiniProjecto2-NoSupervisado/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet

KeyboardInterrupt: 

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.82      0.78      0.80        90
           2       0.35      0.37      0.36        71
           3       0.85      0.80      0.82       178
           4       0.91      0.94      0.93       200
           5       0.93      0.84      0.88       232
           6       0.90      0.92      0.91       135
           7       0.89      0.82      0.85       164
           8       0.51      0.57      0.54        86
           9       0.20      0.25      0.23        55
          10       0.43      0.52      0.47        60
          11       0.47      0.30      0.37       143
          12       0.21      0.20      0.21        64
          13       0.45      0.74      0.56       102
          14       0.26      0.23      0.25        64
          15       0.35      0.30      0.32        81
          16       0.88      0.91      0.90       207

    accuracy                           0.69      1932
   macro avg       0.59   

In [ ]:
joblib.dump(best_model,'model/miniproject2-model.joblib')